# Master Pendulum: Switching and Contact-Aware Modes

This notebook uses the actual `MasterPendulum` class to demonstrate mode selection,
region bands and contact-aware switching.


## Overview

1. Configure a contact-enabled FEM pendulum as the reference model.
2. Instantiate `MasterPendulum` and initialize it.
3. Set contact-aware switching regions with boundary bands.
4. Simulate and inspect synchronization events.


## Learning Goals

- Understand how to configure the `MasterPendulum` for contact-aware switching.
- Understand how region bands prevent chatter without elapsed-time guards.

## Prerequisites

You should be familiar with the FMU, OpenSim, and FEM pendulum components.
This notebook focuses on switching behavior and does not re-derive the models.


In [ ]:
from pathlib import Path
import sys
_repo = Path.cwd()
while _repo != _repo.parent and not (_repo / "pyproject.toml").exists():
    _repo = _repo.parent
sys.path.insert(0, str(_repo))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import opensim as osim

osim.Logger.setLevelString("Warn")

from demos.ControlledPendulum.src.master_pendulum.orchestration.master_pendulum import MasterPendulum
from demos.ControlledPendulum.src.master_pendulum.components.fem import pendulum_config as config
from syssimx import System
from syssimx.system.connection import EventConnection

In [ ]:
import logging

logging.basicConfig(
    level=logging.WARNING,  # keep third-party loggers quiet
    format="[%(module)s] %(levelname)s: %(message)s",
)
# Enable debug output for the syssimx package
logging.getLogger("syssimx").setLevel(logging.DEBUG)

## Configure FEM Parameters

We enable contact so the `MasterPendulum` uses its contact-aware switching logic.
The FEM model acts as the reference for mass/inertia/length synchronization.


In [ ]:
mesh_params = config.MeshParameters()

init_params = config.InitialConditionParameters()
init_params.angular_position_deg = np.rad2deg(0.3) # Initial angle

sim_params = config.SimulationParameters()
sim_params.tau = 0.01
sim_params.t_end = 1
sim_params.with_contact = True
sim_params.use_gravity = True

contact_params = config.ContactParameters()
contact_params.kn = 2e9

anim_params = config.AnimationParameters()
anim_params.animate = False

fem_parameters = {
    'contact_params': contact_params,
    'init_params': init_params,
    'sim_params': sim_params,
    'anim_params': anim_params,
    'mesh_params': mesh_params,
}


## Instantiate MasterPendulum

Initialization will:
- initialize FEM first,
- synchronize parameters to FMU and OpenSim,
- reconcile the configured angle region with the initialized output.


In [ ]:
pendulum = MasterPendulum(name="MasterPendulum", initial_mode="FMU")

pendulum.set_parameters(**{"FEM": fem_parameters})
pendulum.initialize(t0=0.0)

In [ ]:
def wall_contact_event_indicator(pendulum: MasterPendulum):
    theta_wall = 0
    theta_pendulum = pendulum.get_outputs()['theta']
    return theta_pendulum - theta_wall

pendulum.add_event_indicator(name="wall_hit",
                             func=wall_contact_event_indicator,
                             direction=-1)

## Switching Regions and Boundary Bands

We use the default region configuration on absolute angular position.
The `MasterPendulum` assigns FEM to the low-angle region, OpenSim to the intermediate region, and FMU to the high-angle region.

The key reads the wrapper's cached `theta` output, so it does not update FEM grid functions merely to evaluate a switching condition.

Initialization establishes the matching region once; accepted steps switch only at localized boundary events.

```python
    switch_config = MasterPendulumSwitchConfig(
        breakpoints=(0.075, np.deg2rad(15.0)),
        modes=("FEM", "OpenSim", "FMU"),
        bands=(0.005, np.deg2rad(1.0)),
    )
    pendulum = MasterPendulum(switch_config=switch_config)
```

Region boundary bands provide the chatter guard.

In [ ]:
# No elapsed-time guard: valid full-band recrossings must remain switchable.

## Setup System

In [ ]:
system = System(name="PendulumSystem")

system.add_component(pendulum)

event_connection = EventConnection(
    src_comp=pendulum.name, src_port=pendulum.output_specs['wall_hit'].name,
    dst_comp=pendulum.name, dst_port=pendulum.input_specs['omega_invert'].name
)

system.add_event_connection(event_connection)

## Simulate and Record Synchronization Events

We step the system, log the active mode, and inspect `sync_events` to verify
that switching preserves the shared state.


In [ ]:
pendulum.setup_monitoring()
pendulum.display_monitoring()

![Pendulum Monitoring](pendulum_monitoring.png)

**Figure:** Pendulum Monitoring Dashboard



The figure above shows the pendulum monitoring dashboard, which includes:

**Simulation Status**:
- Current time and final time $t$.
- Current time step size $dt$.
- Active mode (FMU, OpenSim, or FEM).

**Input Signals**:
- Current applied torque $\tau$.

**Output Signals**:
- Angular position $\theta$
- Angular velocity $\dot{\theta}=\omega$
- Angular acceleration $\ddot{\theta}=\alpha$

**Stress Visualization**:
- The current normalized Second Piola-Kirchhoff stress distribution in the FEM model, which indicates where the pendulum is experiencing high stress (e.g. due to contact with the wall or due to the torque application).


In [ ]:
system.initialize(t0=0.0)

system.algorithm.record_internal_steps = False
system.algorithm.event_dedup_tol = 5e-4
system.algorithm.tol_time = 1e-5
system.algorithm.tol_value = 5e-5

In [ ]:
print(system.describe())
result = system.run(t0=0.0, tf=sim_params.t_end, dt=sim_params.tau)


In [ ]:
pendulum.fem.animate_stress()

## Visualize Mode Switching

The registered `MasterPendulum` trajectory and event log come from `result`. Per-backend histories are used only to show which internal model supplied each segment.


In [ ]:
t_vals, data = result["MasterPendulum"]
q_vals = data["theta"]
omega_vals = data["omega"]
alpha_vals = data["alpha"]

# Inactive backends are internal to MasterPendulum and are not registered as
# top-level System components, so their diagnostic histories are read directly.
t_vals_fem, data_fem = pendulum.fem.get_history_arrays()
q_vals_fem = data_fem["theta"]
omega_vals_fem = data_fem["omega"]
t_vals_opensim, data_opensim = pendulum.opensim.get_history_arrays()
q_vals_opensim = data_opensim["theta"]
omega_vals_opensim = data_opensim["omega"]
t_vals_fmu, data_fmu = pendulum.fmu.get_history_arrays()
q_vals_fmu = data_fmu["theta"]
omega_vals_fmu = data_fmu["omega"]

event_times = (result.events or {}).get(("MasterPendulum", "wall_hit"), [])
t_event = event_times[0] if event_times else None

print(f"\nDetected {len(event_times)} events")
if t_event is not None:
    print(f"First event at t = {t_event.t:.8f} s")


In [ ]:
mode_styles = {
    "FMU": dict(
        marker="s",
        color='orange',
        linestyle="None",
        markersize=3.2,
        label="FMU",
    ),
    "OpenSim": dict(
        marker="^",
        color='green',
        linestyle="None",
        markersize=3.2,
        label="OpenSim",
    ),
    "FEM": dict(
        marker="o",
        color='blue',
        linestyle="None",
        markersize=3.2,
        label="FEM",
    ),
}

event_style = dict(
    color="#B22222",
    linestyle=":",
    linewidth=1.8,
    label="Event time",
)

wall_style = dict(
    color="0.15",
    linestyle="--",
    linewidth=1.2,
    label="Wall position",
)

grid_style = dict(color="0.85", linewidth=0.7)


In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)

fig.suptitle(
    "Master Pendulum: Event Localization using Hybrid Algorithm",
    fontweight="bold",
    fontsize=14,
    y=1.06,
)

if t_event is not None:
    zoom_window = 0.01
    mask = (t_vals_fem >= t_event.t - zoom_window) & (t_vals_fem <= t_event.t + zoom_window)

# --- Full angle trajectory --------------------------------------------
axs[0, 0].plot(t_vals_fmu, q_vals_fmu, **mode_styles["FMU"])
axs[0, 0].plot(t_vals_opensim, q_vals_opensim, **mode_styles["OpenSim"])
axs[0, 0].plot(t_vals_fem, q_vals_fem, **mode_styles["FEM"])
axs[0, 0].axhline(0, **wall_style)

axs[0, 0].axvline(event_times[0].t, **event_style)
axs[0, 0].axvline(event_times[1].t, **{**event_style, "label": "_nolegend_"})

axs[0, 0].set_ylabel(r"$\theta$ in $\mathrm{rad}$")
axs[0, 0].set_xlabel(r"$t$ in $\mathrm{s}$")

# --- Zoomed angle near event ------------------------------------------
if t_event is not None:
    axs[0, 1].plot(
        t_vals_fem[mask],
        q_vals_fem[mask],
        **{**mode_styles["FEM"], "label": "_nolegend_"},
    )
    axs[0, 1].axhline(0, **{**wall_style, "label": "_nolegend_"})
    axs[0, 1].axvline(t_event.t, **{**event_style, "label": "_nolegend_"})
    axs[0, 1].set_title(rf"Zoomed event at $t={t_event.t:.4f}\,\mathrm{{s}}$")

axs[0, 1].set_ylabel(r"$\theta$ in $\mathrm{rad}$")
axs[0, 1].set_xlabel(r"$t$ in $\mathrm{s}$")

# --- Full angular velocity --------------------------------------------
axs[1, 0].plot(
    t_vals_fmu,
    omega_vals_fmu,
    **{**mode_styles["FMU"], "label": "_nolegend_"},
)
axs[1, 0].plot(
    t_vals_opensim,
    omega_vals_opensim,
    **{**mode_styles["OpenSim"], "label": "_nolegend_"},
)
axs[1, 0].plot(
    t_vals_fem,
    omega_vals_fem,
    **{**mode_styles["FEM"], "label": "_nolegend_"},
)

axs[1, 0].axvline(event_times[0].t, **{**event_style, "label": "_nolegend_"})
axs[1, 0].axvline(event_times[1].t, **{**event_style, "label": "_nolegend_"})

axs[1, 0].set_ylabel(r"$\omega$ in $\mathrm{rad\,s^{-1}}$")
axs[1, 0].set_xlabel(r"$t$ in $\mathrm{s}$")

# --- Zoomed velocity near event ---------------------------------------
if t_event is not None:
    axs[1, 1].plot(
        t_vals_fem[mask],
        omega_vals_fem[mask],
        **{**mode_styles["FEM"], "label": "_nolegend_"},
    )
    axs[1, 1].axvline(t_event.t, **{**event_style, "label": "_nolegend_"})
    axs[1, 1].set_title("Zoomed velocity inversion")

axs[1, 1].set_ylabel(r"$\omega$ in $\mathrm{rad\,s^{-1}}$")
axs[1, 1].set_xlabel(r"$t$ in $\mathrm{s}$")

# --- Shared formatting ------------------------------------------------
for ax in axs.flat:
    ax.grid(True, **grid_style)

# One clean legend from first panel only.
handles, labels = axs[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.04),
    ncol=5,
    frameon=True,
)

plt.show()


## Conclusion

- `System.run()` returns the registered wrapper trajectory and the localized contact events.
- Internal backend histories remain useful diagnostics because inactive models are not top-level system components.
- Region bands prevent threshold chatter while state synchronization keeps the shared pendulum state continuous.

Next: {doc}`../../05_case_study/05_multi_model_switching` applies the same mechanism in the complete controlled-pendulum case study.
